In [2]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input , Embedding , SimpleRNN ,Dense


In [3]:
sentences = [
 "I love this product",
 "This movie made me smile",
 "Service was friendly and quick",
 "Today felt bright and happy",
 "This is the best day",
 "Absolutely fantastic experience",
 "I enjoyed every single moment",
 "Great job, well done",
 "The food tasted delicious",
 "Totally recommend to everyone",
 "Very satisfied with results",
 "This worked better than expected",
 "Amazing quality and value",
 "Such a pleasant surprise",
 "I feel positive about this",
 "I hate this product",
 "This movie bored me",
 "Service was rude and slow",
 "Today was cold and lonely",
 "This is the worst day",
 "Terrible experience overall",
 "I regret buying this",
 "Very disappointed with results",
 "The food tasted awful",
 "Do not recommend this",
 "It broke after one use",
 "Not worth the money",
 "Utterly frustrating and annoying",
 "I feel negative about this",
 "Such a waste of time",
]

labels = [1]*15 + [0]*15
labels = np.array(labels)

In [4]:
labels


array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0])

In [5]:
vocab_size = 2000
tok = Tokenizer(num_words =vocab_size , oov_token = '<OOV>' )
tok.fit_on_texts(sentences)
seqs = tok.texts_to_sequences(sentences)
seqs

[[3, 26, 2, 7],
 [2, 8, 27, 9, 28],
 [10, 6, 29, 4, 30],
 [11, 31, 32, 4, 33],
 [2, 12, 5, 34, 13],
 [35, 36, 14],
 [3, 37, 38, 39, 40],
 [41, 42, 43, 44],
 [5, 15, 16, 45],
 [46, 17, 47, 48],
 [18, 49, 19, 20],
 [2, 50, 51, 52, 53],
 [54, 55, 4, 56],
 [21, 22, 57, 58],
 [3, 23, 59, 24, 2],
 [3, 60, 2, 7],
 [2, 8, 61, 9],
 [10, 6, 62, 4, 63],
 [11, 6, 64, 4, 65],
 [2, 12, 5, 66, 13],
 [67, 14, 68],
 [3, 69, 70, 2],
 [18, 71, 19, 20],
 [5, 15, 16, 72],
 [73, 25, 17, 2],
 [74, 75, 76, 77, 78],
 [25, 79, 5, 80],
 [81, 82, 4, 83],
 [3, 23, 84, 24, 2],
 [21, 22, 85, 86, 87]]

In [6]:
maxlen = max(len(s) for s in seqs)
X = pad_sequences(seqs ,  maxlen = maxlen,padding = 'post')
y = labels

In [7]:
X[0:5]

array([[ 3, 26,  2,  7,  0],
       [ 2,  8, 27,  9, 28],
       [10,  6, 29,  4, 30],
       [11, 31, 32,  4, 33],
       [ 2, 12,  5, 34, 13]], dtype=int32)

In [17]:
embed_dim = 16
rnn_units = 8

inp = Input(shape = (maxlen ,), dtype = "int32" , name = "input")
x = Embedding(input_dim = vocab_size,output_dim =embed_dim , mask_zero = True , name = "embed" )(inp)
rnn = SimpleRNN(units= rnn_units,return_sequences = False,return_state = False , name = "simple_rnn")
x_last = rnn(x)
out = Dense(1,activation = 'sigmoid' , name ='out')(x_last)


In [18]:
model = Model(inputs = inp , outputs= out)
model.compile(optimizer = 'adam' , loss = 'binary_crossentropy' , metrics=['accuracy'])
model.summary()

Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input (InputLayer)  │ (None, 5)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embed (Embedding)   │ (None, 5, 16)     │     32,000 │ input[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_5         │ (None, 5)         │          0 │ input[0][0]       │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ simple_rnn          │ (None, 8)         │        200 │ embed[0][0],      │
│ (SimpleRNN)         │                   │            │ not_equal_5[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ out (Dense)         │ (None, 1)         │          9 │ simple_rnn[0][0]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 32,209 (125.82 KB)

 Trainable params: 32,209 (125.82 KB)

 Non-trainable params: 0 (0.00 B)

In [19]:
model.fit(X,y,
          epochs= 25,
          batch_size = 8,
          verbose = 1
          )

Epoch 1/25


4/4 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - accuracy: 0.4000 - loss: 0.7121
Epoch 2/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.5333 - loss: 0.6933
Epoch 3/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.7000 - loss: 0.6763
Epoch 4/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.7000 - loss: 0.6616
Epoch 5/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.7667 - loss: 0.6459
Epoch 6/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.8667 - loss: 0.6298
Epoch 7/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.9000 - loss: 0.6123
Epoch 8/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.9000 - loss: 0.5942
Epoch 9/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9000 - loss: 0.5759
Epoch 10/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9000 - loss: 0.5561
Epoch 11/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9000 - loss: 0.5344 
Epoch 12/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.9333 - loss: 0.5121
Epoch 13/25
